# Embedded System Experiments
## Lab: Tiling

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

def visualize_systolic_tiling(M, N, K, step):
    TILE_SIZE = 8
    MAX_DIM = 32

    M, N, K = max(1, M), max(1, N), max(1, K)
    grid_m, grid_n, grid_k = int(np.ceil(M/8)), int(np.ceil(N/8)), int(np.ceil(K/8))

    # Nested Loop Logic: i -> j -> k
    idx_m = (step // (grid_n * grid_k)) % grid_m
    idx_n = (step // grid_k) % grid_n
    idx_k = step % grid_k

    # Figure 1: Global Memory View
    fig1, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
    def draw_mem(ax, r, c, active_r, active_c, title, color):
        ax.set_aspect('equal')
        ax.add_patch(plt.Rectangle((0, 0), MAX_DIM, MAX_DIM, color='#eeeeee', zorder=0))
        ax.add_patch(plt.Rectangle((0, 0), c, r, edgecolor='black', facecolor='white', lw=1.5, zorder=1))
        ax.add_patch(plt.Rectangle((active_c*8, active_r*8), 8, 8, facecolor=color, alpha=0.6, zorder=2))
        ax.set_xlim(0, MAX_DIM); ax.set_ylim(MAX_DIM, 0)
        ax.set_title(title, fontweight='bold')
        ax.set_xticks(np.arange(0, 33, 8)); ax.set_yticks(np.arange(0, 33, 8))
        ax.grid(True, ls=':', alpha=0.5)

    draw_mem(ax1, M, K, idx_m, idx_k, "Matrix A Memory", "#4CC9F0")
    draw_mem(ax2, K, N, idx_k, idx_n, "Matrix B Memory", "#F72585")
    draw_mem(ax3, M, N, idx_m, idx_n, "Matrix C Memory", "#4361EE")
    plt.show()

    # Figure 2: Hardware Internal View (The 8x8 Systolic Array)
    fig2, (hw_a, hw_b, hw_sum) = plt.subplots(1, 3, figsize=(15, 5))

    def draw_hw(ax, rows, cols, active_r, active_c, title, color, is_sum=False):
        ax.set_aspect('equal')
        # Draw 8x8 PE Grid
        for x in range(8):
            for y in range(8):
                # Padding Logic: If the global index exceeds matrix size, it's a 0-pad
                global_y = active_r * 8 + y
                global_x = active_c * 8 + x

                is_padding = (global_y >= rows) or (global_x >= cols)
                face = "#ffcccc" if is_padding else color # Reddish for padding
                alpha = 0.2 if is_padding else 0.8

                rect = plt.Rectangle((x, y), 1, 1, facecolor=face, edgecolor='black', alpha=alpha)
                ax.add_patch(rect)
                if is_padding:
                    ax.text(x+0.5, y+0.5, '0', ha='center', va='center', fontsize=8, color='red')

        ax.set_xlim(0, 8); ax.set_ylim(8, 0)
        ax.set_title(title, fontweight='bold')
        ax.set_xticks(range(9)); ax.set_yticks(range(9))

    # Partial Sum Progress (How many K-tiles have we merged?)
    progress = (idx_k + 1) / grid_k

    draw_hw(hw_a, M, K, idx_m, idx_k, "Array Input: A-Tile", "#4CC9F0")
    draw_hw(hw_b, K, N, idx_k, idx_n, "Array Input: B-Tile", "#F72585")

    # Matrix C HW (Showing accumulation progress)
    hw_sum.set_aspect('equal')
    for x in range(8):
        for y in range(8):
            global_y, global_x = idx_m * 8 + y, idx_n * 8 + x
            is_valid = (global_y < M) and (global_x < N)
            color = "#4361EE" if is_valid else "#eeeeee"
            # Fill level based on idx_k (Partial Sum progress)
            ax_sum_rect = plt.Rectangle((x, y), 1, 1, facecolor='white', edgecolor='black')
            hw_sum.add_patch(ax_sum_rect)
            if is_valid:
                fill = plt.Rectangle((x, y + (1-progress)), 1, progress, facecolor=color, alpha=0.9)
                hw_sum.add_patch(fill)

    hw_sum.set_xlim(0, 8); hw_sum.set_ylim(8, 0)
    hw_sum.set_title(f"Accumulator (C-Tile)\nPartial Sum: {idx_k+1}/{grid_k}", fontweight='bold')
    plt.show()

# UI Setup
m_s = widgets.IntSlider(value=20, min=1, max=32, description='M:')
n_s = widgets.IntSlider(value=20, min=1, max=32, description='N:')
k_s = widgets.IntSlider(value=18, min=1, max=32, description='K:')
step_s = widgets.IntSlider(value=0, min=0, max=0, description='Step:')

def update_steps(*args):
    total = int(np.ceil(m_s.value/8) * np.ceil(n_s.value/8) * np.ceil(k_s.value/8))
    step_s.max = max(0, total - 1)

for s in [m_s, n_s, k_s]: s.observe(update_steps, 'value')
update_steps()

ui = widgets.VBox([widgets.HBox([m_s, n_s, k_s]), step_s])
out = widgets.interactive_output(visualize_systolic_systolic_tiling if 'visualize_systolic_systolic_tiling' in locals() else visualize_systolic_tiling,
                                 {'M': m_s, 'N': n_s, 'K': k_s, 'step': step_s})
display(ui, out)

Output()